In [ ]:
%cd ..
%pwd

In [ ]:

# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2

# Core Python and data tools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

# Mesa model components
import mesa
from mesa import model
from mesa import agent
from traffic.model.traffic_model import TrafficModel
from traffic.agents.vehicle_agent import VehicleAgent
from traffic.agents.road_segment_agent import RoadSegmentAgent

# Custom utility modules (aliased for clarity)
import traffic.utils.unit_conversion_utils as uc
import traffic.utils.analysis_utils as au
import traffic.utils.animation_utils as anim

# Visualization and display (Jupyter-specific)
from IPython.display import display, HTML

print(mesa.__version__)
pd.set_option("display.max_rows", None)  


# Load external data

In [ ]:
# road stuff
road_gdf = gpd.read_parquet("data/roads/hw210_sl_and_curvs.parquet")
# Expected counts
ecs_df = pd.read_csv('data/vehicle_counts/expected_counts_seconds.csv')

#au.plot_colored_road(road_gdf.loc[road_gdf.curvature>15], 'curvature')
au.plot_colored_road(road_gdf.iloc[::1], 'curvature')

# Single Model Run

In [ ]:
%%time
tm = TrafficModel(
    # meta perams
    road_gdf=road_gdf, ecs_df=ecs_df, max_steps=3000, batchrun=False, collect_every_n=10,
    # car centric perams
    start_hr=7, traffic_percentile=50, max_persons=5000,
    # canyon closure
    canyon_closures={'closure_step': [1], 'duration': [1], 'road_section': [119]},  # ie no closures
    #bus centric perams
    bus_interval=5, bus_capacity=30, car_preference=.5,
    crashes_per_100k_vmt_input=0
)

tm.run_model()
print(f'Model ran for {tm.steps} steps')


In [ ]:
# #Warm up a bit so vehicles exist, then profile a shorter window
# for _ in range(2500): tm.step()
# %prun -s cumtime [tm.step() for _ in range(1000)]

# Analysis

In [ ]:
finished_agents = au.finished_agents_summary_df(tm, plots=True)


In [ ]:
# process the finished_agents data 
vehicles_full = au.vehicle_agent_data_time_series(tm, plots=True)
model_ts = au.model_data_time_series(tm)

au.plot_speed_delta(vehicles_full, model_ts)

# if speed limit break and prevent pass dont appear it is likley because of an outdated enviroment. Traffic_model calls generate.py and passes the model as a perameter. Sometimes this dosent work properly and the agents are not registered with the model. When not registered with the model they cant be seen by other agents (occures in get_next_agent()).

In [ ]:
#vehicles_full.loc[vehicles_full.status == 'crash']
#vehicles_full.loc[(vehicles_full.AgentID.between(405, 407)) & (vehicles_full.Step.between(300,305))]

# vehicles_full.loc[(vehicles_full.AgentID==406) & (vehicles_full.Step.between(190,220))]

In [ ]:
# currently usefull for volume_by_section, speed_by_section, density_by_section
#au.plot_section_trends(model_ts, col='volume_by_section', window=1)

# currently usefull for speed_change, speed_change_mps2, speed, speed_mps, gap_m, ideal_gap_m
#au.plot_mean_feature(vehicles_full, 'gap_m')

# Animations

In [ ]:
# run the animation
# looking at one car
issue_car_id =  604
issue_step = 0

In [ ]:
anim.animate_traffic(vehicles_full, road_gdf, interval=100, step_skip=2, watch=None, zoom=20)

In [ ]:
anim.animate_traffic_with_speed_delta_highlight(vehicles_full, road_gdf, model_ts, interval=100, step_skip=3, watch=None, zoom=20)

In [ ]:
anim.animate_relative_distance(vehicle_df=vehicles_full, agent_id=700, distance_behind=100, color_by='status')

# Issue Car Analysis

In [ ]:
au.plot_single_car_driving_actions(vehicles_full, issue_car_id)

In [ ]:
step_range=(500,600)
au.plot_agent_trajectories(vehicles_full, [issue_car_id, issue_car_id+1], 'speed',step_range)